# 이미지 생성 통합 테스트 빌드 (2026 Ver.)

이 노트북은 **OpenAI (DALL-E 3)**와 **Google (Nano Banana)** API를 사용하여 장르 맞춤형 이미지를 생성합니다.

### API 발급처 정보
- **DALL-E 3**: [OpenAI Platform](https://platform.openai.com/)에서 발급 (`OPENAI_API_KEY`)
- **Nano Banana**: [Google AI Studio](https://aistudio.google.com/) 또는 [Vertex AI](https://cloud.google.com/vertex-ai)에서 발급 (`GOOGLE_API_KEY`)

---

## 0. 환경 변수 및 라이브러리 설정
`.env` 파일에서 키를 읽어오고 사용할 라이브러리를 로드합니다.

In [ ]:
import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# .env 파일 로드
load_dotenv()

# 발급처별 API 키 매핑 확인
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY") 

def check_keys():
    if not OPENAI_API_KEY:
        print("⚠️ OpenAI API 키가 설정되지 않았습니다.")
    else:
        print("✅ OpenAI API 키 확인됨")
        
    if not GOOGLE_API_KEY:
        print("⚠️ Google (Nano Banana) API 키가 설정되지 않았습니다.")
    else:
        print("✅ Google API 키 확인됨")

check_keys()

# 클라이언트 초기화
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

## 1. 전처리 데이터셋 로드
기존에 생성한 장르/배경 메타데이터를 불러옵니다.

In [ ]:
OUTPUT_DIR = Path("./output")

def load_indexes():
    scene_path = OUTPUT_DIR / "scene_metadata_index.json"
    try:
        with open(scene_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print("⚠️ 전처리 파일이 없습니다. 기본 설명으로 진행합니다.")
        return []

scene_index = load_indexes()

## 2. 장르별 시각적 스타일 엔진 (10대 장르 + 고전)
각 장르에 최적화된 그림체 설명을 정의합니다.

In [ ]:
STYLE_CONFIG = {
    "드라마": "Cinematic realism, warm lighting, emotional depth, photography style.",
    "멜로/로맨스": "Soft pastel aesthetic, dreamy watercolor, ethereal glow, romantic atmosphere.",
    "스릴러": "Gritty noir, high contrast, low-key lighting, tense and dark atmosphere.",
    "미스터리": "Foggy, mysterious mood, deep shadows, cinematic suspense, detailed textures.",
    "공포(호러)": "Grotesque shadows, desaturated tones, eerie misty environment, horror movie style.",
    "액션": "Dynamic motion lines, high contrast, vivid and sharp, dramatic camera angles.",
    "코미디": "Bright and punchy colors, vibrant digital art, clean and cheerful vibe.",
    "SF": "Futuristic cyberpunk, neon accents, metallic surfaces, high-tech aesthetic.",
    "전쟁": "Gritty battlefield realism, dusty sepia tones, epic scale, cinematic grit.",
    "판타지": "Epic high-fantasy, magical particle effects, mystical creatures, vibrant concept art.",
    
    # 고전 세계관 전용 (Prepend)
    "CLASSIC_KR": "Traditional Korean ink wash painting (Sumukhwa) mixed with modern digital detail.",
    "CLASSIC_CN": "Wuxia fantasy art style, sweeping brush strokes, traditional Chinese color palette.",
    "CLASSIC_JP": "Ukiyo-e inspired bold lines, mystical folklore aesthetic, Japanese traditional art."
}

def build_style_prompt(genre, background):
    """장르와 배경을 조합하여 최종 화풍 프롬프트를 만듭니다."""
    style_parts = []
    
    # 고전 배경 우선순위
    if any(k in background for k in ["조선", "한국", "설화"]): style_parts.append(STYLE_CONFIG["CLASSIC_KR"])
    elif "중국" in background or "무협" in background: style_parts.append(STYLE_CONFIG["CLASSIC_CN"])
    elif "일본" in background: style_parts.append(STYLE_CONFIG["CLASSIC_JP"])
    
    # 장르 스타일 추가
    style_parts.append(STYLE_CONFIG.get(genre, "Detailed digital illustration"))
    
    return ", ".join(style_parts)

## 3. OpenAI DALL-E 3 연동
OpenAI에서 제공하는 최신 이미지 생성 모델을 호출합니다.

In [ ]:
def generate_with_dalle(prompt):
    if not openai_client:
        return "❌ OpenAI API 키 누락"
    
    try:
        print(f"🚀 DALL-E 3 생성 중: {prompt[:50]}...")
        response = openai_client.images.generate(
            model="dall-e-3",
            prompt=prompt,
            size="1024x1792",
            n=1
        )
        return response.data[0].url
    except Exception as e:
        return f"❌ DALL-E 오류: {e}"

## 4. Google Nano Banana 연동
Google의 초고속 이미지 생성 모델인 Nano Banana (Gemini 기반)를 호출합니다.

In [ ]:
def generate_with_nano_banana(prompt):
    """Google Nano Banana API 호출 시뮬레이션 (2026 Ver.)"""
    if not GOOGLE_API_KEY:
        return "❌ Google API 키 누락"
    
    try:
        print(f"🍌 Nano Banana 생성 중: {prompt[:50]}...")
        # 실제 Google Cloud / Vertex AI 엔드포인트 호출 가정
        # (2026년 기준 Gemini Image API 규격에 따름)
        headers = {"Authorization": f"Bearer {GOOGLE_API_KEY}", "Content-Type": "application/json"}
        payload = {"model": "nano-banana-pro", "prompt": prompt, "aspect_ratio": "16:9"}
        
        # response = requests.post("https://api.google.com/image/generate", json=payload, headers=headers)
        # return response.json().get('url')
        return "https://storage.googleapis.com/nano-banana-results/sample.png"
    except Exception as e:
        return f"❌ Nano Banana 오류: {e}"

## 5. 통합 이미지 생성기
설정된 모든 요소를 결합하여 최종 결과물을 도출합니다.

In [ ]:
def create_world_image(user_input, genre, background, provider="dalle"):
    # 1. 스타일 구성
    style = build_style_prompt(genre, background)
    
    # 2. 데이터셋 기반 묘사 보강
    reference = ""
    if scene_index:
        matches = [s for s in scene_index if genre in s.get('genre', [])]
        if matches: reference = f" (Context hint: {matches[0].get('storyline', '')})"
    
    # 3. 최종 프롬프트 빌드
    final_prompt = f"A cinematic scene of {user_input}.{reference} Style: {style}. 8k, detailed masterpiece."
    
    # 4. API 실행
    if provider == "dalle":
        return generate_with_dalle(final_prompt)
    else:
        return generate_with_nano_banana(final_prompt)

# ── 테스트 실행 ──────────────────────────────────────────────────────────
test_url = create_world_image("달빛 아래 서 있는 구미호", "판타지", "조선")
print(f"\n✨ 최종 이미지 결과: {test_url}")